# StairKid RL Unified Training
This notebook clones Git source, verifies external assets separately, runs one fail-closed precheck, and invokes the repository trainer. It contains no physics, reward, environment, or PPO implementation.


In [ ]:
TRAIN_TARGET = "v3"  # exactly: v3 or r4
REPO_URL = "https://github.com/GameToy21452244/stairkid-rl.git"
GIT_REF = "main"  # use an exact commit SHA or tag for reproducible formal runs
OUTPUT_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/StairKidRL/runs"
TRAINING_ASSET_SOURCE_DIR = None  # e.g. a Drive directory containing the pinned R4 bundle
MODEL_ASSET_SOURCE_DIR = None  # required only for R4 smoke unless its canonical model is cached
RESUME = False
RESUME_CHECKPOINT = None
RESUME_METADATA = None
TRAINING_MODE = "smoke"  # precheck, smoke, or full
AUTHORIZATION = ""  # full requires AUTHORIZE_STAIRKID_FULL_TRAINING
assert TRAIN_TARGET in {"v3", "r4"}
assert TRAINING_MODE in {"precheck", "smoke", "full"}


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
WORKDIR = Path("/content/stairkid-rl")
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "checkout", GIT_REF], cwd=WORKDIR, check=True)
RESOLVED_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[rl]"], cwd=WORKDIR, check=True)
import stair_agent
print(f"REPOSITORY={REPO_URL}")
print(f"COMMIT={RESOLVED_COMMIT}")
print(f"TRAIN_TARGET={TRAIN_TARGET}")
print(f"PYTHON={sys.version.split()[0]}")
print("PACKAGE_IMPORT=PASS")


In [ ]:
import stable_baselines3
import torch
print(f"TORCH={torch.__version__}")
print(f"CUDA_AVAILABLE={torch.cuda.is_available()}")
print(f"CUDA_VERSION={torch.version.cuda}")
print(f"GPU_NAME={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"SB3_VERSION={stable_baselines3.__version__}")
if TRAINING_MODE == "full" and not torch.cuda.is_available():
    print("WARNING=Full training is configured for CPU provenance; Colab GPU is unavailable.")


In [ ]:
if OUTPUT_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path(DRIVE_OUTPUT_ROOT)
else:
    OUTPUT_ROOT = WORKDIR / "runs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if TRAIN_TARGET == "r4":
    asset_cmd = [sys.executable, "scripts/fetch_training_assets.py", "--target", "r4"]
    if TRAINING_ASSET_SOURCE_DIR:
        asset_cmd += ["--source-dir", TRAINING_ASSET_SOURCE_DIR]
    subprocess.run(asset_cmd, cwd=WORKDIR, check=True)
    if TRAINING_MODE == "smoke":
        model_cmd = [sys.executable, "scripts/fetch_models.py", "--model", "r4"]
        if MODEL_ASSET_SOURCE_DIR:
            model_cmd += ["--source-dir", MODEL_ASSET_SOURCE_DIR]
        subprocess.run(model_cmd, cwd=WORKDIR, check=True)


In [ ]:
precheck_cmd = [sys.executable, "scripts/train.py", "--target", TRAIN_TARGET, "--mode", "precheck", "--output", str(OUTPUT_ROOT)]
precheck = subprocess.run(precheck_cmd, cwd=WORKDIR, text=True, capture_output=True)
print(precheck.stdout)
if precheck.returncode != 0:
    print(precheck.stderr)
    raise RuntimeError("STAIRKID_TRAINING_PRECHECK=FAIL")
assert "STAIRKID_TRAINING_PRECHECK=PASS" in precheck.stdout
print("REPO_CHECKOUT=PASS")
print(f"GIT_COMMIT={RESOLVED_COMMIT}")
print("WORKTREE_CLEAN=PASS")
print("CONFIG_VALID=PASS")
print(f"TRAIN_TARGET={TRAIN_TARGET}")
print("OBSERVATION_SPACE=(268,)")
print("ACTION_SPACE=Discrete(3)")
print(f"OUTPUT_DIR={OUTPUT_ROOT}")
print("STAIRKID_TRAINING_PRECHECK=PASS")


In [ ]:
if TRAINING_MODE == "full" and AUTHORIZATION != "AUTHORIZE_STAIRKID_FULL_TRAINING":
    raise RuntimeError("FULL_TRAINING_NOT_AUTHORIZED")
train_cmd = [sys.executable, "scripts/train.py", "--target", TRAIN_TARGET, "--mode", TRAINING_MODE, "--output", str(OUTPUT_ROOT)]
if TRAINING_MODE == "full":
    train_cmd += ["--authorization", AUTHORIZATION]
if RESUME:
    if not RESUME_CHECKPOINT or not RESUME_METADATA:
        raise RuntimeError("RESUME_CHECKPOINT_AND_METADATA_REQUIRED")
    train_cmd += ["--resume", RESUME_CHECKPOINT, "--resume-metadata", RESUME_METADATA]
subprocess.run(train_cmd, cwd=WORKDIR, check=True)
